# Layered, multiphase screening: band-pass + repeat + velocity → ML → LLM

A `ScreeningPipeline` layers several cheap detectors into a combined risk, then escalates only the
risky items: a **band-pass** range check, a stateful **repeat** screen (the same thing N times in a
row), and a **velocity** screen (N events in a window). Flagged items go to an ML classifier and
then an LLM (`ALLOW`/`REVIEW`/`BLOCK`). Two domains below — payments and sensor telemetry — share
the same core.

Runs with no model server (built-in lexicon classifier); set `ANTHROPIC_API_KEY` for the LLM tier.
Prereqs: `mvn -DskipTests package`, `pip install -e python/`.

In [1]:
import os, pathlib, subprocess
from collections import Counter
repo_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
if 'AGENTIC_FLINK_JAR' not in os.environ:
    jars = [j for j in sorted((repo_root/'target').glob('agentic-flink-*.jar'))
            if 'original' not in j.name and 'sources' not in j.name]
    os.environ['AGENTIC_FLINK_JAR'] = str(jars[-1])
cp = repo_root/'target'/'runtime-classpath.txt'
if not cp.exists():
    subprocess.run(['mvn','-q','dependency:build-classpath',f'-Dmdep.outputFile={cp}',
                    '-Dmdep.includeScope=runtime'], cwd=repo_root, check=True)
import agentic_flink as af
af.start_jvm(extra_jars=[j for j in cp.read_text().strip().split(':') if j])
from agentic_flink import jclass
Duration = jclass('java.time.Duration')
ScreeningPipeline = jclass('org.agentic.flink.screening.ScreeningPipeline')
ScreenItem = jclass('org.agentic.flink.screening.ScreenItem')
BandPass = jclass('org.agentic.flink.screening.BandPassDetector')
Repeat = jclass('org.agentic.flink.screening.RepeatDetector')
Velocity = jclass('org.agentic.flink.screening.VelocityDetector')
Lexicon = jclass('org.agentic.flink.inference.LexiconInferenceConnection')
InferenceSetup = jclass('org.agentic.flink.inference.InferenceSetup')
key = os.environ.get('ANTHROPIC_API_KEY')
lex_setup = InferenceSetup.builder().withModelName('lexicon').withModelUri('lexicon://default').build()
print('ready —', 'LLM tier: Claude' if key else 'LLM tier disabled (no ANTHROPIC_API_KEY)')

ready — LLM tier disabled (no ANTHROPIC_API_KEY)


## Domain 1 — payment screening

Band-pass on amount (expect < $1000), repeat (3 in a row on an account), velocity (5 in a minute),
then the lexicon classifier + optional LLM.

In [2]:
b = (ScreeningPipeline.builder()
       .addDetector(BandPass(0.0, 1000.0, 0.6))
       .addDetector(Repeat(3, 0.8))
       .addDetector(Velocity(5, Duration.ofMinutes(1), 0.7))
       .withClassifier(Lexicon(), lex_setup)
       .withReviewThreshold(0.5).withBlockThreshold(2.0))
if key: b = b.withClaude(key)
payments_kb = b.build()

# (account, amount, merchant, ts_ms)
payments = [
    ('acct-1', 12.5,  'Coffee Hut',       0),
    ('acct-1', 80.0,  'Grocers',          60000),
    ('acct-2', 4999.0,'Wire Transfer Co', 5000),     # out-of-band amount
    ('acct-3', 250.0, 'GiftCardz',        1000),     # repeat 3x in a row ->
    ('acct-3', 250.0, 'GiftCardz',        2000),
    ('acct-3', 250.0, 'GiftCardz',        3000),
    ('acct-4', 9.0,   'AppStore',         0),         # velocity burst ->
    ('acct-4', 9.0,   'AppStore',         5000),
    ('acct-4', 9.0,   'AppStore',         10000),
    ('acct-4', 9.0,   'AppStore',         15000),
    ('acct-4', 9.0,   'AppStore',         20000),
]
funnel = Counter()
for acct, amt, merch, ts in payments:
    item = ScreenItem.of(acct, float(amt), merch, ts)
    r = payments_kb.screen(item)
    funnel[str(r.decidedBy)] += 1
    phases = ','.join(sorted({str(s.phase()) for s in r.fired})) or '-'
    print(f'{str(r.decidedBy):<5} {r.verdict:<6} risk={r.combinedRisk:.2f} [{phases:<22}] | {acct} ${amt:.2f} @ {merch}')
print('\nFunnel:', dict(funnel))

RULES ALLOW  risk=0.00 [-                     ] | acct-1 $12.50 @ Coffee Hut
RULES ALLOW  risk=0.00 [-                     ] | acct-1 $80.00 @ Grocers
ML    REVIEW risk=1.10 [BAND_PASS,CLASSIFIER  ] | acct-2 $4999.00 @ Wire Transfer Co
RULES ALLOW  risk=0.00 [-                     ] | acct-3 $250.00 @ GiftCardz
RULES ALLOW  risk=0.00 [-                     ] | acct-3 $250.00 @ GiftCardz
ML    REVIEW risk=0.80 [CLASSIFIER,REPEAT     ] | acct-3 $250.00 @ GiftCardz
RULES ALLOW  risk=0.00 [-                     ] | acct-4 $9.00 @ AppStore
RULES ALLOW  risk=0.00 [-                     ] | acct-4 $9.00 @ AppStore
ML    REVIEW risk=0.80 [CLASSIFIER,REPEAT     ] | acct-4 $9.00 @ AppStore
ML    REVIEW risk=0.80 [CLASSIFIER,REPEAT     ] | acct-4 $9.00 @ AppStore
ML    REVIEW risk=1.50 [CLASSIFIER,REPEAT,VELOCITY] | acct-4 $9.00 @ AppStore

Funnel: {'RULES': 6, 'ML': 5}


## Domain 2 — sensor telemetry band-pass

Same core, different config: band-pass on the operating range, repeat = stuck sensor (identical
reading 4×), velocity = flapping. No ML/LLM tier (numeric).

In [3]:
telemetry_kb = (ScreeningPipeline.builder()
    .addDetector(BandPass(-40.0, 85.0, 0.7))
    .addDetector(Repeat.sameValue(4, 0.8))
    .addDetector(Velocity(5, Duration.ofSeconds(1), 0.6))
    .withReviewThreshold(0.5).withBlockThreshold(1.5)
    .build())

readings = [
    ('temp-1', 21.4, 0), ('temp-1', 22.1, 1000), ('temp-1', 98.7, 2000),   # spike out-of-band
    ('temp-2', 20.0, 0), ('temp-2', 20.0, 1000), ('temp-2', 20.0, 2000), ('temp-2', 20.0, 3000),  # stuck
    ('temp-3', 19.0, 0), ('temp-3', 25.0, 200), ('temp-3', 18.0, 400), ('temp-3', 27.0, 600), ('temp-3', 17.0, 800),  # flapping
]
for sid, val, ts in readings:
    r = telemetry_kb.screen(ScreenItem.of(sid, float(val), sid, ts))
    phases = ','.join(sorted({str(s.phase()) for s in r.fired})) or '-'
    print(f'{str(r.decidedBy):<5} {r.verdict:<6} risk={r.combinedRisk:.2f} [{phases:<22}] | {sid} = {val}')

RULES ALLOW  risk=0.00 [-                     ] | temp-1 = 21.4
RULES ALLOW  risk=0.00 [-                     ] | temp-1 = 22.1
RULES REVIEW risk=0.70 [BAND_PASS             ] | temp-1 = 98.7
RULES ALLOW  risk=0.00 [-                     ] | temp-2 = 20.0
RULES ALLOW  risk=0.00 [-                     ] | temp-2 = 20.0
RULES ALLOW  risk=0.00 [-                     ] | temp-2 = 20.0
RULES REVIEW risk=0.80 [REPEAT                ] | temp-2 = 20.0
RULES ALLOW  risk=0.00 [-                     ] | temp-3 = 19.0
RULES ALLOW  risk=0.00 [-                     ] | temp-3 = 25.0
RULES ALLOW  risk=0.00 [-                     ] | temp-3 = 18.0
RULES ALLOW  risk=0.00 [-                     ] | temp-3 = 27.0
RULES REVIEW risk=0.60 [VELOCITY              ] | temp-3 = 17.0


## What happened

Each item accumulates weighted signals across phases; below the review threshold it's `ALLOW`ed,
in the review band it escalates (ML → LLM), and overwhelming risk is `BLOCK`ed outright. The repeat
and velocity detectors are **stateful** — they fire only once enough history accrues for a key.
Swap `LexiconInferenceConnection` for `DjlInferenceConnection` for a trained classifier; the
streaming versions are `PaymentScreeningExample` / `TelemetryScreeningExample`.